# Materi 3 — Memaksimalkan Tools AI untuk Data Internal
**Pelatihan Gen AI Integrated Data Science**

---

## Tujuan pembelajaran
1. Memahami dua jenis data internal — **terstruktur** (database) dan **tidak terstruktur** (dokumen) — beserta pendekatan AI yang cocok.
2. Melakukan **profiling & pemeriksaan kualitas data** sebelum data dipakai AI.
3. Menerapkan **klasifikasi data dan penyamaran PII** (data pribadi) sebelum data dikirim ke LLM.
4. Membangun **Text-to-SQL** yang aman: LLM menulis SQL, Python memvalidasi & menjalankan dalam mode *read-only*.
5. Membangun **RAG (Retrieval-Augmented Generation)** sederhana untuk menjawab pertanyaan dari SOP internal dengan sitasi.
6. Menggabungkan keduanya menjadi **asisten data internal**.

```
            ┌──────────── Pertanyaan pengguna ────────────┐
            ▼                                             ▼
   Data TERSTRUKTUR (SQL)                       Data TIDAK TERSTRUKTUR (dokumen)
   Text-to-SQL → validasi → DB read-only        RAG: cari potongan relevan → LLM + sitasi
            └───────────── Jawaban + sumber ──────────────┘
      Lapisan pengaman: klasifikasi data · penyamaran PII · log audit
```
> Database, konsumen, dan isi dokumen SOP di notebook ini **fiktif** untuk keperluan latihan.

## Langkah 0 — Library
| Library | Kegunaan |
|---|---|
| `sqlite3` | Database SQL ringan (bawaan Python) sebagai tiruan sistem internal (DMS/ERP). |
| `pandas` | Membaca hasil query (`pd.read_sql`) dan profiling. |
| `re` | *Regular expression* untuk mendeteksi & menyamarkan nomor HP, email, NIK. |
| `hashlib` | Membuat token pseudonim yang konsisten dari nama konsumen. |
| `sklearn TfidfVectorizer` | Mengubah teks dokumen menjadi vektor untuk pencarian (retrieval) RAG. |
| `cosine_similarity` | Mengukur kemiripan pertanyaan dengan potongan dokumen. |

In [1]:
import hashlib, os, sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 90)
DB_PATH = "ahm_internal_demo.db"
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)            # mulai dari database bersih setiap notebook dijalankan

## Langkah 1 — Membangun database internal tiruan
Empat tabel yang meniru sistem internal:

| Tabel | Isi | Klasifikasi |
|---|---|---|
| `dealer` | ID, nama, wilayah, kelas dealer | Internal |
| `penjualan` | No. rangka, tanggal, dealer, model, harga, id konsumen | Internal |
| `konsumen` | Nama, no. HP, email, NIK, kota | **Rahasia – data pribadi** |
| `klaim_garansi` | ID klaim, no. rangka, tanggal, komponen, biaya, status | Internal |

Beberapa **masalah kualitas data** sengaja ditanam (duplikat, biaya negatif, nilai kosong, tanggal tidak logis) untuk latihan profiling.

In [2]:
rng = np.random.default_rng(3)
WILAYAH = ["Jabodetabek", "Jawa Barat", "Jawa Tengah", "Jawa Timur", "Sumatera", "Kalimantan"]
MODEL = ["BeAT", "Vario 125", "Scoopy", "PCX 160", "ADV 160", "CBR150R"]
NAMA_DEPAN = ["Andi", "Budi", "Citra", "Dewi", "Eko", "Fitri", "Gilang", "Hana", "Irfan", "Joko", "Kartika", "Lina"]
NAMA_BELAKANG = ["Saputra", "Wijaya", "Lestari", "Pratama", "Hidayat", "Nugroho", "Santoso", "Rahma"]

# --- dealer ---
dealer = pd.DataFrame({
    "dealer_id": [f"D{i:03d}" for i in range(1, 41)],
    "nama_dealer": [f"Dealer Contoh {i}" for i in range(1, 41)],
    "wilayah": rng.choice(WILAYAH, 40),
    "kelas": rng.choice(["Wing", "Reguler"], 40, p=[.3, .7]),
})

# --- konsumen (berisi PII fiktif) ---
N = 3000
konsumen = pd.DataFrame({
    "id_konsumen": [f"K{i:05d}" for i in range(1, N + 1)],
    "nama": [f"{rng.choice(NAMA_DEPAN)} {rng.choice(NAMA_BELAKANG)}" for _ in range(N)],
    "no_hp": [f"08{rng.integers(11, 99)}{rng.integers(10_000_000, 99_999_999)}" for _ in range(N)],
    "nik": [f"3{rng.integers(10**14, 10**15 - 1)}" for _ in range(N)],
    "kota": rng.choice(["Jakarta", "Bandung", "Semarang", "Surabaya", "Medan", "Balikpapan"], N),
})
konsumen["email"] = konsumen["nama"].str.lower().str.replace(" ", ".") + konsumen.index.astype(str) + "@contoh.id"

# --- penjualan ---
penjualan = pd.DataFrame({
    "no_rangka": [f"MH1{rng.integers(10**9, 10**10 - 1)}" for _ in range(N)],
    "tanggal": (pd.Timestamp("2025-01-01") + pd.to_timedelta(rng.integers(0, 540, N), unit="D")).strftime("%Y-%m-%d"),
    "dealer_id": rng.choice(dealer["dealer_id"], N),
    "model": rng.choice(MODEL, N, p=[.34, .24, .15, .12, .08, .07]),
    "harga_otr": rng.integers(18_000, 40_000, N) * 1000,
    "id_konsumen": konsumen["id_konsumen"],
})

# --- klaim garansi (dengan masalah kualitas yang ditanam) ---
M = 350
idx = rng.choice(N, M, replace=False)
klaim = pd.DataFrame({
    "id_klaim": [f"CLM-{i:05d}" for i in range(1, M + 1)],
    "no_rangka": penjualan.loc[idx, "no_rangka"].values,
    "tanggal_klaim": (pd.to_datetime(penjualan.loc[idx, "tanggal"]).values
                      + pd.to_timedelta(rng.integers(10, 300, M), unit="D")),
    "komponen": rng.choice(["Injektor", "Kiprok", "Starter", "Kampas kopling", "Seal shock", "ECU", "Pompa bensin"], M),
    "biaya": rng.integers(150, 2_500, M) * 1000,
    "status": rng.choice(["Disetujui", "Ditolak", "Proses"], M, p=[.7, .15, .15]),
})
klaim["tanggal_klaim"] = pd.to_datetime(klaim["tanggal_klaim"]).dt.strftime("%Y-%m-%d")
klaim.loc[rng.choice(M, 5, replace=False), "biaya"] *= -1                  # biaya negatif
klaim.loc[rng.choice(M, 8, replace=False), "komponen"] = None              # komponen kosong
klaim.loc[klaim.index[:2], "tanggal_klaim"] = "2024-06-01"                 # klaim sebelum motor terjual
klaim = pd.concat([klaim, klaim.sample(3, random_state=1)], ignore_index=True)   # 3 baris duplikat

# --- simpan ke SQLite ---
with sqlite3.connect(DB_PATH) as con:
    for nama, tabel in {"dealer": dealer, "konsumen": konsumen, "penjualan": penjualan, "klaim_garansi": klaim}.items():
        tabel.to_sql(nama, con, index=False)
        print(f"Tabel {nama:<14} → {len(tabel):>5} baris")

Tabel dealer         →    40 baris
Tabel konsumen       →  3000 baris
Tabel penjualan      →  3000 baris
Tabel klaim_garansi  →   353 baris


**Penjelasan kode**
- `[f"D{i:03d}" for i in range(1, 41)]` — *list comprehension* untuk membuat ID dealer D001–D040.
- `rng.integers(10**14, 10**15 - 1)` — angka acak 15 digit untuk NIK fiktif (awalan "3" → total 16 digit).
- `.str.lower().str.replace(" ", ".")` — operasi teks vektor pada kolom pandas (`.str` accessor) untuk membentuk email.
- `rng.choice(N, M, replace=False)` — memilih 350 motor **berbeda** yang mengajukan klaim.
- `.values` — mengambil array NumPy murni agar penjumlahan tanggal tidak terganggu penyelarasan indeks pandas.
- Blok "masalah kualitas": `*= -1` membuat biaya negatif, `= None` membuat nilai kosong, `klaim.sample(3)` + `concat` membuat duplikat.
- `tabel.to_sql(nama, con, index=False)` — menulis DataFrame menjadi tabel SQL. `with sqlite3.connect(...) as con:` otomatis melakukan *commit* setelah blok selesai.

## Langkah 2 — Lapisan LLM

In [3]:
# ============================================================
# LAPISAN LLM: satu "pintu" untuk semua pemanggilan Gen AI
# ============================================================
import os, json, re, time
import requests

# --- Konfigurasi (bisa diubah lewat environment variable) ---
LLM_MODE     = os.getenv("LLM_MODE", "simulasi")            # "simulasi" | "anthropic" | "openai_compatible"
LLM_MODEL    = os.getenv("LLM_MODEL", "claude-sonnet-5-5")  # nama model sesuai penyedia
LLM_BASE_URL = os.getenv("LLM_BASE_URL", "http://localhost:11434/v1")  # untuk openai_compatible (mis. Ollama lokal)


class LLMClient:
    """Pembungkus (wrapper) agar kode analisis tidak terikat pada satu vendor AI."""

    def __init__(self, mode=LLM_MODE, model=LLM_MODEL, simulator=None):
        self.mode = mode            # cara memanggil model
        self.model = model          # nama model
        self.simulator = simulator  # fungsi pengganti LLM untuk mode 'simulasi' (offline)
        self.log = []               # jejak audit setiap pemanggilan

    def generate(self, prompt, system="Anda adalah analis data yang teliti.",
                 temperature=0.0, max_tokens=1200):
        t0 = time.time()
        if self.mode == "anthropic":
            teks = self._panggil_anthropic(prompt, system, temperature, max_tokens)
        elif self.mode == "openai_compatible":
            teks = self._panggil_openai_compatible(prompt, system, temperature, max_tokens)
        else:
            teks = self.simulator(prompt, system)
        # catat metadata (BUKAN isi prompt) untuk audit & monitoring biaya
        self.log.append({
            "waktu": time.strftime("%Y-%m-%d %H:%M:%S"),
            "mode": self.mode,
            "panjang_prompt": len(prompt),
            "panjang_jawaban": len(teks),
            "durasi_detik": round(time.time() - t0, 3),
        })
        return teks

    def _panggil_anthropic(self, prompt, system, temperature, max_tokens):
        r = requests.post(
            "https://api.anthropic.com/v1/messages",
            headers={"x-api-key": os.environ["ANTHROPIC_API_KEY"],
                     "anthropic-version": "2023-06-01",
                     "content-type": "application/json"},
            json={"model": self.model, "max_tokens": max_tokens, "temperature": temperature,
                  "system": system, "messages": [{"role": "user", "content": prompt}]},
            timeout=120)
        r.raise_for_status()                      # hentikan jika HTTP error (401, 429, dst.)
        return r.json()["content"][0]["text"]     # ambil teks jawaban pertama

    def _panggil_openai_compatible(self, prompt, system, temperature, max_tokens):
        r = requests.post(
            f"{LLM_BASE_URL}/chat/completions",
            headers={"Authorization": f"Bearer {os.getenv('LLM_API_KEY', 'tidak-perlu')}"},
            json={"model": self.model, "temperature": temperature, "max_tokens": max_tokens,
                  "messages": [{"role": "system", "content": system},
                               {"role": "user", "content": prompt}]},
            timeout=120)
        r.raise_for_status()
        return r.json()["choices"][0]["message"]["content"]


def ambil_json(teks):
    """Mengambil objek/array JSON dari jawaban LLM yang kadang dibungkus ```json ...```
    atau diawali kalimat pembuka."""
    teks = re.sub(r"```(?:json)?", "", teks)
    kandidat = [i for i in (teks.find("{"), teks.find("[")) if i != -1]
    if not kandidat:
        raise ValueError("Tidak ditemukan JSON pada jawaban LLM")
    awal = min(kandidat)
    akhir = teks.rfind("}" if teks[awal] == "{" else "]")
    return json.loads(teks[awal:akhir + 1])

print("Mode LLM aktif:", LLM_MODE)

Mode LLM aktif: simulasi


### Penjelasan kode — Lapisan LLM

Sel ini adalah **fondasi** semua pemanggilan Gen AI di notebook. Semua langkah analisis hanya memanggil `llm.generate(prompt)`, sehingga kita bisa berganti vendor/model tanpa mengubah logika analisis.

| Bagian | Fungsi | Mengapa penting |
|---|---|---|
| `LLM_MODE = os.getenv(...)` | Membaca mode dari *environment variable*; bawaan `"simulasi"`. | Notebook tetap berjalan di kelas tanpa internet/API key. |
| `LLM_MODEL`, `LLM_BASE_URL` | Nama model & alamat server (untuk model lokal seperti Ollama). | Data sensitif bisa diproses model **lokal** di jaringan internal. |
| `class LLMClient` | *Wrapper* yang menyatukan 3 cara pemanggilan. | Prinsip *separation of concerns*: analisis ≠ koneksi API. |
| `generate()` | Memilih jalur (`anthropic` / `openai_compatible` / simulator), lalu mencatat log. | Satu titik kontrol untuk audit, biaya, dan *error handling*. |
| `self.log.append({...})` | Menyimpan waktu, mode, panjang prompt/jawaban, durasi. | Isi prompt **tidak** disimpan → mengurangi risiko kebocoran data. |
| `requests.post(...)` | Memanggil REST API. `headers` berisi kunci API; `json` berisi model, `system`, dan `messages`. | Tanpa SDK tambahan — cukup library `requests`. |
| `temperature=0.0` | Mengatur keacakan jawaban; 0 = paling konsisten. | Untuk analisis data kita butuh hasil yang **dapat diulang**. |
| `r.raise_for_status()` | Memunculkan error bila status HTTP bukan 2xx. | Error terlihat jelas, tidak diam-diam menghasilkan data kosong. |
| `ambil_json()` | Membersihkan pagar ```` ```json ```` lalu memotong dari `{`/`[` pertama sampai penutup terakhir, kemudian `json.loads`. | LLM kadang menambahkan kalimat pembuka; fungsi ini membuat parsing **tangguh**. |

**Cara memakai model sungguhan** (opsional, sebelum menjalankan notebook):
```bash
# Claude (Anthropic)
export LLM_MODE=anthropic
export ANTHROPIC_API_KEY=sk-ant-...
# atau server kompatibel OpenAI (mis. Ollama lokal / gateway internal perusahaan)
export LLM_MODE=openai_compatible
export LLM_BASE_URL=http://localhost:11434/v1
export LLM_MODEL=llama3.1
```
> ⚠️ **Kebijakan data:** jangan mengirim data rahasia/pribadi ke layanan AI publik tanpa persetujuan sesuai kebijakan perusahaan. Mode **simulasi** memakai fungsi aturan sederhana (*rule-based*) yang meniru format jawaban LLM, sehingga seluruh alur bisa dipelajari secara offline.

### Simulator untuk notebook ini
Meniru LLM untuk tiga tugas: menulis **catatan kualitas data**, menulis **SQL** dari pertanyaan, dan **menjawab dari potongan dokumen** (RAG). Pada pertanyaan tentang data pribadi, simulator sengaja menulis SQL yang mengambil kolom PII — untuk menguji lapisan pengaman kita.

In [4]:
def _antara(teks, awal, akhir):
    return teks.rsplit(awal, 1)[1].split(akhir, 1)[0].strip()


SQL_POLA = {
    "klaim garansi dan total biaya per model": """
SELECT p.model, COUNT(*) AS jumlah_klaim, SUM(k.biaya) AS total_biaya
FROM klaim_garansi k JOIN penjualan p ON k.no_rangka = p.no_rangka
WHERE k.status = 'Disetujui' AND k.biaya > 0
GROUP BY p.model ORDER BY jumlah_klaim DESC""",
    "rasio klaim tertinggi": """
WITH jual AS (SELECT dealer_id, COUNT(*) AS unit FROM penjualan GROUP BY dealer_id),
     klm AS (SELECT p.dealer_id, COUNT(DISTINCT k.id_klaim) AS klaim
             FROM klaim_garansi k JOIN penjualan p ON k.no_rangka = p.no_rangka GROUP BY p.dealer_id)
SELECT d.dealer_id, d.wilayah, j.unit, k.klaim, ROUND(100.0 * k.klaim / j.unit, 1) AS rasio_klaim_persen
FROM jual j JOIN klm k ON j.dealer_id = k.dealer_id JOIN dealer d ON d.dealer_id = j.dealer_id
ORDER BY rasio_klaim_persen DESC LIMIT 5""",
    "komponen": """
SELECT komponen, COUNT(*) AS jumlah, ROUND(AVG(biaya)) AS rata_biaya
FROM klaim_garansi WHERE komponen IS NOT NULL AND biaya > 0
GROUP BY komponen ORDER BY jumlah DESC""",
    "nomor hp": """
SELECT c.nama, c.no_hp, k.komponen FROM klaim_garansi k
JOIN penjualan p ON k.no_rangka = p.no_rangka JOIN konsumen c ON c.id_konsumen = p.id_konsumen""",
    "hapus": "DELETE FROM klaim_garansi WHERE status = 'Ditolak'",
}


def simulator_nb03(prompt, system=None):
    if "TUGAS: CATATAN_KUALITAS" in prompt:
        cek = ambil_json(_antara(prompt, "HASIL_CEK:", "AKHIR_CEK"))
        gagal = [c for c in cek if c["jumlah_masalah"] > 0]
        baris = [f"- **{c['pemeriksaan']}** pada `{c['kolom']}`: {c['jumlah_masalah']} baris. "
                 f"Saran: {c['saran']}" for c in gagal]
        return ("Catatan untuk pemilik data (Warranty Admin):\n"
                f"Dari {len(cek)} pemeriksaan, {len(gagal)} menemukan masalah:\n" + "\n".join(baris) +
                "\nMohon perbaikan di sistem sumber agar laporan klaim bulanan tidak bias.")
    if "TUGAS: TEXT_TO_SQL" in prompt:
        q = _antara(prompt, "PERTANYAAN:", "AKHIR_PERTANYAAN").lower()
        for kunci, sql in SQL_POLA.items():
            if all(k in q for k in kunci.split()[:2]) or kunci in q:
                return "```sql" + sql + "\n```"
        return "```sql\nSELECT COUNT(*) AS jumlah FROM penjualan\n```"
    if "TUGAS: JAWAB_DARI_DOKUMEN" in prompt:
        q = _antara(prompt, "PERTANYAAN:", "AKHIR_PERTANYAAN").lower()
        blok = _antara(prompt, "KONTEKS:", "AKHIR_KONTEKS")
        if not blok or blok == "(kosong)":
            return "Informasi tersebut tidak ditemukan dalam dokumen internal yang tersedia."
        kata_q = {w for w in re.findall(r"\w+", q) if len(w) > 3}
        kandidat = []
        for bagian in blok.split("[SUMBER: ")[1:]:
            sumber, isi = bagian.split("]", 1)
            for kal in re.split(r"(?<=\.)\s+", isi.strip()):
                skor = len(kata_q & set(re.findall(r"\w+", kal.lower())))
                kandidat.append((skor, kal.strip(), sumber))
        skor, kalimat, sumber = max(kandidat, key=lambda x: x[0])   # seri → kalimat paling awal
        if skor == 0:
            return "Informasi tersebut tidak ditemukan dalam dokumen internal yang tersedia."
        return f"{kalimat} [{sumber}]"
    return "Tugas tidak dikenali."


llm = LLMClient(simulator=simulator_nb03)

**Penjelasan kode**
- `SQL_POLA` — kamus pola pertanyaan → SQL yang akan "ditulis" simulator. Perhatikan pola `"nomor hp"` (mengambil PII) dan `"hapus"` (perintah `DELETE`) — keduanya **harus diblokir** oleh validator nanti.
- `re.findall(r"\w+", q)` — mengambil semua kata; `re.split(r"(?<=\.)\s+", isi)` memecah paragraf menjadi kalimat setelah tanda titik (*lookbehind* `(?<=\.)`).
- Pada tugas RAG, simulator memilih kalimat dengan irisan kata terbanyak terhadap pertanyaan lalu **menyertakan sumbernya** — perilaku yang kita minta dari LLM sungguhan melalui prompt.

## Langkah 3 — Kualitas data dulu: *profiling* otomatis
*Garbage in, garbage out.* Sebelum data internal dipakai AI, jalankan pemeriksaan standar. Hasil pemeriksaan (bukan data mentah) kemudian dikirim ke LLM untuk dituliskan menjadi catatan yang mudah dipahami pemilik data.

In [5]:
with sqlite3.connect(DB_PATH) as con:
    k = pd.read_sql("SELECT * FROM klaim_garansi", con)
    p = pd.read_sql("SELECT no_rangka, tanggal FROM penjualan", con)

gabung = k.merge(p, on="no_rangka", how="left")
cek = [
    {"pemeriksaan": "Duplikat baris", "kolom": "semua",
     "jumlah_masalah": int(k.duplicated().sum()), "saran": "hapus duplikat, tambahkan unique key pada id_klaim"},
    {"pemeriksaan": "Nilai kosong", "kolom": "komponen",
     "jumlah_masalah": int(k["komponen"].isna().sum()), "saran": "jadikan field wajib pada form klaim"},
    {"pemeriksaan": "Nilai tidak logis (negatif)", "kolom": "biaya",
     "jumlah_masalah": int((k["biaya"] < 0).sum()), "saran": "validasi input biaya > 0"},
    {"pemeriksaan": "Urutan tanggal tidak logis", "kolom": "tanggal_klaim",
     "jumlah_masalah": int((pd.to_datetime(gabung["tanggal_klaim"]) < pd.to_datetime(gabung["tanggal"])).sum()),
     "saran": "klaim tidak boleh sebelum tanggal penjualan"},
    {"pemeriksaan": "Referensi tidak ditemukan", "kolom": "no_rangka",
     "jumlah_masalah": int(gabung["tanggal"].isna().sum()), "saran": "cek integritas relasi ke penjualan"},
]
tabel_cek = pd.DataFrame(cek)
display(tabel_cek)

catatan = llm.generate(f"""TUGAS: CATATAN_KUALITAS
Anda Data Steward. Tulis catatan singkat untuk pemilik data berdasarkan HASIL_CEK.
Sebut hanya pemeriksaan yang bermasalah, jumlah barisnya, dan saran perbaikan.
HASIL_CEK: {json.dumps(cek, ensure_ascii=False)} AKHIR_CEK""")
print(catatan)

,pemeriksaan,kolom,jumlah_masalah,saran
0,Duplikat baris,semua,3,"hapus duplikat, tambahkan unique key pada id_klaim"
1,Nilai kosong,komponen,8,jadikan field wajib pada form klaim
2,Nilai tidak logis (negatif),biaya,5,validasi input biaya > 0
3,Urutan tanggal tidak logis,tanggal_klaim,2,klaim tidak boleh sebelum tanggal penjualan
4,Referensi tidak ditemukan,no_rangka,0,cek integritas relasi ke penjualan


Catatan untuk pemilik data (Warranty Admin):
Dari 5 pemeriksaan, 4 menemukan masalah:
- **Duplikat baris** pada `semua`: 3 baris. Saran: hapus duplikat, tambahkan unique key pada id_klaim
- **Nilai kosong** pada `komponen`: 8 baris. Saran: jadikan field wajib pada form klaim
- **Nilai tidak logis (negatif)** pada `biaya`: 5 baris. Saran: validasi input biaya > 0
- **Urutan tanggal tidak logis** pada `tanggal_klaim`: 2 baris. Saran: klaim tidak boleh sebelum tanggal penjualan
Mohon perbaikan di sistem sumber agar laporan klaim bulanan tidak bias.


**Penjelasan kode**
- `pd.read_sql(query, con)` — menjalankan SQL dan langsung menghasilkan DataFrame.
- `k.merge(p, on="no_rangka", how="left")` — *left join*: semua klaim dipertahankan, ditambah tanggal penjualan. Jika tidak ada pasangan, `tanggal` bernilai kosong → dipakai untuk cek integritas relasi.
- `k.duplicated().sum()` — menghitung baris yang identik dengan baris sebelumnya.
- `(pd.to_datetime(a) < pd.to_datetime(b)).sum()` — perbandingan tanggal vektor; `True` dihitung sebagai 1.
- Setiap pemeriksaan disimpan sebagai dict dengan **saran perbaikan** → hasilnya bisa menjadi tiket perbaikan data.
- LLM hanya menerima **hasil cek** (angka & nama kolom) — tidak ada satu pun data klaim yang dikirim.

## Langkah 4 — Klasifikasi data & penyamaran PII
Aturan emas: **data pribadi tidak dikirim ke layanan AI eksternal.** Kita membuat dua lapisan:
1. **Gerbang kebijakan** — tabel level klasifikasi vs tujuan (AI publik / AI internal).
2. **Penyamaran (*masking*) & pseudonim** — nama diganti token konsisten, no. HP & email disamarkan, NIK dihapus, dan pola PII di teks bebas ikut dibersihkan.

In [6]:
KEBIJAKAN = pd.DataFrame({
    "level":           ["Publik", "Internal", "Rahasia", "Sangat Rahasia"],
    "contoh":          ["brosur, harga OTR publik", "penjualan agregat, SOP", "data dealer rinci, biaya klaim",
                        "data pribadi konsumen (nama, NIK, HP)"],
    "AI_publik":       ["boleh", "boleh jika disamarkan/agregat", "tidak", "tidak"],
    "AI_internal":     ["boleh", "boleh", "boleh (akses terbatas)", "hanya setelah pseudonim/masking"],
})
display(KEBIJAKAN)


def boleh_kirim(level, tujuan):
    aturan = KEBIJAKAN.set_index("level").loc[level, "AI_publik" if tujuan == "publik" else "AI_internal"]
    return not aturan.startswith("tidak"), aturan


print(boleh_kirim("Sangat Rahasia", "publik"))
print(boleh_kirim("Internal", "publik"))

,level,contoh,AI_publik,AI_internal
0,Publik,"brosur, harga OTR publik",boleh,boleh
1,Internal,"penjualan agregat, SOP",boleh jika disamarkan/agregat,boleh
2,Rahasia,"data dealer rinci, biaya klaim",tidak,boleh (akses terbatas)
3,Sangat Rahasia,"data pribadi konsumen (nama, NIK, HP)",tidak,hanya setelah pseudonim/masking


(False, 'tidak')
(True, 'boleh jika disamarkan/agregat')


**Penjelasan kode**
- `KEBIJAKAN` — tabel keputusan (*decision table*) yang menerjemahkan kebijakan perusahaan menjadi data. **Sesuaikan dengan kebijakan resmi perusahaan Anda**; tabel ini hanya contoh.
- `set_index("level").loc[level, kolom]` — mencari sel berdasarkan level & tujuan.
- `boleh_kirim()` mengembalikan tuple `(boleh?, aturan)` sehingga program bisa menolak otomatis sekaligus menampilkan alasannya.

In [7]:
POLA_HP    = re.compile(r"\b(08\d{2})\d{4,6}(\d{3})\b")
POLA_EMAIL = re.compile(r"\b([\w.])[\w.]*@([\w.]+)\b")
POLA_NIK   = re.compile(r"\b\d{16}\b")
GARAM = "ganti-dengan-rahasia-perusahaan"      # 'salt' agar token tidak bisa ditebak ulang


def pseudonim(nilai, awalan="KONS"):
    return f"{awalan}-{hashlib.sha256((GARAM + str(nilai)).encode()).hexdigest()[:8]}"


def samarkan_teks(teks):
    teks = POLA_NIK.sub("[NIK]", teks)
    teks = POLA_HP.sub(r"\1****\2", teks)
    teks = POLA_EMAIL.sub(r"\1***@\2", teks)
    return teks


def samarkan_df(data):
    d = data.copy()
    if "nama" in d:  d["nama"] = d["nama"].map(pseudonim)
    if "no_hp" in d: d["no_hp"] = d["no_hp"].map(samarkan_teks)
    if "email" in d: d["email"] = d["email"].map(samarkan_teks)
    return d.drop(columns=[c for c in ["nik"] if c in d])


with sqlite3.connect(DB_PATH) as con:
    contoh = pd.read_sql("SELECT nama, no_hp, email, nik, kota FROM konsumen LIMIT 4", con)
print("SEBELUM:"); display(contoh)
print("SESUDAH:"); display(samarkan_df(contoh))

catatan_cs = f"Pelanggan a.n. {contoh.loc[0, 'nama']} (HP {contoh.loc[0, 'no_hp']}, NIK {contoh.loc[0, 'nik']}) komplain starter."
print("Teks bebas asli     :", catatan_cs)
print("Teks bebas disamarkan:", samarkan_teks(catatan_cs).replace(contoh.loc[0, "nama"], pseudonim(contoh.loc[0, "nama"])))

SEBELUM:


,nama,no_hp,email,nik,kota
0,Joko Wijaya,081596531442,joko.wijaya0@contoh.id,3700377130917724,Balikpapan
1,Andi Nugroho,082326912370,andi.nugroho1@contoh.id,3889863753824079,Medan
2,Citra Santoso,086272209205,citra.santoso2@contoh.id,3267271206958513,Medan
3,Dewi Hidayat,084538357953,dewi.hidayat3@contoh.id,3370332001795062,Bandung


SESUDAH:


,nama,no_hp,email,kota
0,KONS-3f6f295b,0815****442,j***@contoh.id,Balikpapan
1,KONS-aeae7bdf,0823****370,a***@contoh.id,Medan
2,KONS-dc35db63,0862****205,c***@contoh.id,Medan
3,KONS-21e009eb,0845****953,d***@contoh.id,Bandung


Teks bebas asli     : Pelanggan a.n. Joko Wijaya (HP 081596531442, NIK 3700377130917724) komplain starter.
Teks bebas disamarkan: Pelanggan a.n. KONS-3f6f295b (HP 0815****442, NIK [NIK]) komplain starter.


**Penjelasan kode**
- `re.compile(...)` — mengompilasi pola regex sekali agar dipakai berulang secara efisien.
  - `POLA_HP`: `(08\d{2})` menangkap 4 digit awal, `\d{4,6}` bagian tengah, `(\d{3})` 3 digit akhir. `\b` = batas kata.
  - `POLA_EMAIL`: menangkap huruf pertama dan domain, sisanya disamarkan.
  - `POLA_NIK`: tepat 16 digit berurutan.
- `.sub(r"\1****\2", teks)` — mengganti pola dengan grup tangkapan pertama (`\1`), bintang, lalu grup kedua (`\2`): `0812****789`.
- `pseudonim()` — hash **SHA-256** dari *salt* + nama → token `KONS-xxxxxxxx`. Nama yang sama selalu menjadi token yang sama (analisis tetap bisa menghitung per konsumen), tetapi nama asli tidak dapat dibaca. *Salt* disimpan rahasia agar token tidak bisa ditebak dengan mencoba nama umum.
- `d.drop(columns=["nik"])` — kolom yang tidak diperlukan untuk analisis **dihapus**, bukan sekadar disamarkan (prinsip *data minimization*).

## Langkah 5 — Text-to-SQL yang aman
Pengguna bisnis bertanya dalam bahasa sehari-hari, LLM menulis SQL. Tiga lapisan pengaman:
1. **Skema terkurasi** — LLM hanya diberi tabel/kolom yang diizinkan (kolom PII tidak dideskripsikan).
2. **Validator SQL** — hanya `SELECT`/`WITH`, satu pernyataan, tanpa kata kunci berbahaya, tanpa kolom PII, otomatis ditambah `LIMIT`.
3. **Koneksi read-only** — walaupun validator kebobolan, database menolak perubahan.

In [8]:
KOLOM_PII = {"nama", "no_hp", "email", "nik"}


def deskripsi_skema(db_path, sembunyikan=KOLOM_PII):
    baris = []
    with sqlite3.connect(db_path) as con:
        tabel = [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table'")]
        for t in tabel:
            kolom = [(r[1], r[2]) for r in con.execute(f"PRAGMA table_info({t})") if r[1] not in sembunyikan]
            baris.append(f"- {t}(" + ", ".join(f"{n} {tp}" for n, tp in kolom) + ")")
    return "\n".join(baris)


SKEMA = deskripsi_skema(DB_PATH)
print(SKEMA)

- dealer(dealer_id TEXT, nama_dealer TEXT, wilayah TEXT, kelas TEXT)
- konsumen(id_konsumen TEXT, kota TEXT)
- penjualan(no_rangka TEXT, tanggal TEXT, dealer_id TEXT, model TEXT, harga_otr INTEGER, id_konsumen TEXT)
- klaim_garansi(id_klaim TEXT, no_rangka TEXT, tanggal_klaim TEXT, komponen TEXT, biaya INTEGER, status TEXT)


**Penjelasan kode**
- `sqlite_master` — tabel sistem SQLite yang berisi daftar tabel. Setiap database punya padanannya (mis. `INFORMATION_SCHEMA` di SQL Server/PostgreSQL).
- `PRAGMA table_info(tabel)` — mengembalikan daftar kolom; indeks `r[1]` = nama kolom, `r[2]` = tipe.
- Kolom PII **tidak dimasukkan** ke deskripsi skema → LLM tidak "tahu" kolom tersebut ada. Ini lapisan pertama (*least privilege*).

In [9]:
KATA_TERLARANG = r"\b(INSERT|UPDATE|DELETE|DROP|ALTER|CREATE|REPLACE|ATTACH|DETACH|PRAGMA|VACUUM|TRUNCATE)\b"


def validasi_sql(sql, batas=200):
    s = sql.strip().rstrip(";")
    if ";" in s:
        raise PermissionError("Lebih dari satu pernyataan SQL")
    if not re.match(r"^\s*(SELECT|WITH)\b", s, flags=re.I):
        raise PermissionError("Hanya query SELECT yang diizinkan")
    if re.search(KATA_TERLARANG, s, flags=re.I):
        raise PermissionError("Mengandung perintah pengubah data")
    kolom_terpakai = {w.lower() for w in re.findall(r"\b\w+\b", s)}
    if kolom_terpakai & KOLOM_PII:
        raise PermissionError(f"Mengakses kolom data pribadi: {sorted(kolom_terpakai & KOLOM_PII)}")
    if not re.search(r"\bLIMIT\s+\d+", s, flags=re.I):
        s += f"\nLIMIT {batas}"
    return s


def jalankan_readonly(sql):
    uri = f"file:{DB_PATH}?mode=ro"                     # buka database dalam mode baca saja
    with sqlite3.connect(uri, uri=True) as con:
        return pd.read_sql(sql, con)


def tanya_database(pertanyaan):
    prompt = f"""TUGAS: TEXT_TO_SQL
Anda ahli SQL (SQLite). Tulis SATU query SELECT untuk menjawab pertanyaan.
Gunakan hanya tabel & kolom berikut:
{SKEMA}
Catatan bisnis: klaim valid = status 'Disetujui' dan biaya > 0.
Kembalikan hanya blok ```sql```.
PERTANYAAN: {pertanyaan} AKHIR_PERTANYAAN"""
    jawaban = llm.generate(prompt)
    sql = re.search(r"```sql\s*(.*?)```", jawaban, flags=re.S).group(1)
    print(f"Q: {pertanyaan}")
    try:
        sql_aman = validasi_sql(sql)
    except PermissionError as e:
        print(f"⛔ DITOLAK: {e}\n   SQL usulan: {' '.join(sql.split())[:110]}...\n")
        return None
    print("SQL:", " ".join(sql_aman.split())[:160], "...")
    return jalankan_readonly(sql_aman)


display(tanya_database("Berapa jumlah klaim garansi dan total biaya per model?"))
display(tanya_database("Dealer mana dengan rasio klaim tertinggi?"))
display(tanya_database("Komponen apa yang paling sering diklaim dan berapa rata-rata biayanya?"))
tanya_database("Tampilkan nama dan nomor HP konsumen yang mengajukan klaim")
tanya_database("Hapus klaim yang ditolak")

Q: Berapa jumlah klaim garansi dan total biaya per model?
SQL: SELECT p.model, COUNT(*) AS jumlah_klaim, SUM(k.biaya) AS total_biaya FROM klaim_garansi k JOIN penjualan p ON k.no_rangka = p.no_rangka WHERE k.status = 'Diset ...


,model,jumlah_klaim,total_biaya
0,BeAT,81,104609000
1,Vario 125,52,74013000
2,Scoopy,35,41971000
3,CBR150R,23,37199000
4,PCX 160,22,34490000
5,ADV 160,15,15171000


Q: Dealer mana dengan rasio klaim tertinggi?
SQL: WITH jual AS (SELECT dealer_id, COUNT(*) AS unit FROM penjualan GROUP BY dealer_id), klm AS (SELECT p.dealer_id, COUNT(DISTINCT k.id_klaim) AS klaim FROM klaim_ ...


,dealer_id,wilayah,unit,klaim,rasio_klaim_persen
0,D028,Jawa Timur,69,16,23.2
1,D026,Jawa Tengah,86,14,16.3
2,D030,Sumatera,74,12,16.2
3,D013,Jawa Timur,81,13,16.0
4,D021,Jawa Tengah,76,12,15.8


Q: Komponen apa yang paling sering diklaim dan berapa rata-rata biayanya?
SQL: SELECT komponen, COUNT(*) AS jumlah, ROUND(AVG(biaya)) AS rata_biaya FROM klaim_garansi WHERE komponen IS NOT NULL AND biaya > 0 GROUP BY komponen ORDER BY juml ...


,komponen,jumlah,rata_biaya
0,Pompa bensin,58,1259103.0
1,Injektor,57,1265474.0
2,Seal shock,52,1532481.0
3,Starter,47,1303234.0
4,Kampas kopling,44,1357000.0
5,ECU,43,1320349.0
6,Kiprok,39,1326077.0


Q: Tampilkan nama dan nomor HP konsumen yang mengajukan klaim
⛔ DITOLAK: Mengakses kolom data pribadi: ['nama', 'no_hp']
   SQL usulan: SELECT c.nama, c.no_hp, k.komponen FROM klaim_garansi k JOIN penjualan p ON k.no_rangka = p.no_rangka JOIN kon...

Q: Hapus klaim yang ditolak
⛔ DITOLAK: Hanya query SELECT yang diizinkan
   SQL usulan: DELETE FROM klaim_garansi WHERE status = 'Ditolak'...



**Penjelasan kode**
- `validasi_sql()` — pemeriksaan berlapis:
  - `rstrip(";")` lalu cek `";" in s` → menolak *multiple statement* (mis. `SELECT ...; DROP TABLE ...`).
  - `re.match(r"^\s*(SELECT|WITH)\b", ..., flags=re.I)` → wajib diawali `SELECT` atau `WITH` (CTE), tidak peka huruf besar/kecil.
  - `KATA_TERLARANG` → regex `\b(...)\b` menolak kata kunci pengubah data/struktur.
  - Irisan kata dengan `KOLOM_PII` → menolak query yang menyentuh data pribadi, **walaupun** LLM "menebak" nama kolomnya.
  - Menambah `LIMIT` agar query tidak menarik jutaan baris.
- `sqlite3.connect("file:...?mode=ro", uri=True)` — koneksi **read-only**. Pada database perusahaan, padanannya adalah akun database khusus dengan hak `SELECT` saja pada *view* tertentu.
- `re.search(r"```sql\s*(.*?)```", ..., flags=re.S).group(1)` — mengambil isi blok SQL dari jawaban LLM.
- Query `WITH jual AS (...)` pada rasio klaim adalah **CTE** (*Common Table Expression*): subquery bernama agar SQL panjang lebih mudah dibaca. `COUNT(DISTINCT ...)` menghindari duplikat klaim terhitung ganda.
- Dua pertanyaan terakhir menunjukkan pengaman bekerja: permintaan data pribadi dan perintah `DELETE` **ditolak sebelum menyentuh database**.

## Langkah 6 — RAG: menjawab dari dokumen internal
**Retrieval-Augmented Generation** = (1) cari potongan dokumen yang relevan, (2) sisipkan ke prompt, (3) LLM menjawab **hanya** dari potongan tersebut dan menyebut sumbernya.

Kita memakai **TF-IDF** sebagai mesin pencari (ringan, tanpa internet). Di produksi, TF-IDF bisa diganti *embedding model* + *vector database* — konsepnya sama.

In [10]:
DOKUMEN = {
    "SOP-GAR-01 Klaim Garansi": """Klaim garansi diajukan dealer melalui sistem klaim paling lambat 14 hari kalender setelah perbaikan selesai. Dokumen wajib meliputi foto komponen rusak, nomor rangka, dan kilometer saat kerusakan.
Klaim garansi ditolak apabila kerusakan disebabkan modifikasi, kecelakaan, atau penggunaan suku cadang tidak asli. Komponen yang diganti wajib disimpan di dealer selama 60 hari untuk kemungkinan audit.
Klaim dengan biaya di atas Rp2.000.000 memerlukan persetujuan Area Service Manager sebelum perbaikan dilakukan.""",
    "SOP-PDI-02 Pre-Delivery Inspection": """Setiap unit baru wajib menjalani Pre-Delivery Inspection sebelum diserahkan kepada konsumen. Pemeriksaan meliputi tekanan ban, level oli, fungsi rem, kelistrikan, dan kekencangan baut.
Hasil PDI dicatat pada checklist dan ditandatangani mekanik serta kepala bengkel. Unit yang gagal PDI tidak boleh diserahkan sampai perbaikan selesai dan diperiksa ulang.""",
    "KEB-AI-03 Kebijakan Penggunaan Gen AI": """Karyawan dilarang memasukkan data pribadi konsumen, data keuangan rahasia, atau kode sumber internal ke layanan AI publik. Data internal boleh diproses dengan AI hanya dalam bentuk agregat atau setelah disamarkan.
Setiap keluaran AI yang dipakai untuk keputusan bisnis wajib ditinjau oleh manusia yang bertanggung jawab. Penggunaan AI untuk proses baru harus didaftarkan ke tim Data Governance.""",
    "PAN-SRV-04 Panduan Servis Berkala": """Servis berkala gratis dilakukan sesuai Kartu Perawatan Berkala pada kelipatan kilometer yang ditentukan buku servis. Penggantian oli mesin dianjurkan setiap 2.000 sampai 4.000 km tergantung kondisi pemakaian.
Untuk motor matic, pemeriksaan CVT termasuk v-belt dan roller dilakukan setiap servis berkala. Konsumen disarankan melakukan booking servis melalui aplikasi agar waktu tunggu lebih singkat.""",
    "SOP-CR-05 Eskalasi Keluhan Pelanggan": """Keluhan dengan urgensi tinggi, yaitu yang menyangkut keselamatan atau motor tidak dapat digunakan, wajib ditindaklanjuti dalam 1x24 jam. Keluhan urgensi sedang ditindaklanjuti maksimal 3 hari kerja.
Jika keluhan tidak selesai di tingkat dealer, kasus dieskalasi ke Main Dealer lalu ke tim Customer Relations pusat. Setiap eskalasi dicatat beserta akar masalah dan tindakan korektif.""",
}
print("Isi dokumen fiktif untuk latihan. Jumlah dokumen:", len(DOKUMEN))


def pecah_dokumen(teks, maks_kata=60, tumpang=15):
    """Memecah teks menjadi potongan ±maks_kata kata dengan tumpang-tindih agar konteks tidak terpotong."""
    kata = teks.split()
    potongan, mulai = [], 0
    while mulai < len(kata):
        potongan.append(" ".join(kata[mulai:mulai + maks_kata]))
        if mulai + maks_kata >= len(kata):
            break
        mulai += maks_kata - tumpang
    return potongan


chunks = pd.DataFrame([{"sumber": judul.split()[0], "judul": judul, "isi": pot}
                       for judul, teks in DOKUMEN.items() for pot in pecah_dokumen(teks)])
print("Jumlah potongan (chunk):", len(chunks))
chunks.head(4)

Isi dokumen fiktif untuk latihan. Jumlah dokumen: 5
Jumlah potongan (chunk): 6


,sumber,judul,isi
0,SOP-GAR-01,SOP-GAR-01 Klaim Garansi,Klaim garansi diajukan dealer melalui sistem klaim paling lambat 14 hari kalender sete...
1,SOP-GAR-01,SOP-GAR-01 Klaim Garansi,disimpan di dealer selama 60 hari untuk kemungkinan audit. Klaim dengan biaya di atas ...
2,SOP-PDI-02,SOP-PDI-02 Pre-Delivery Inspection,Setiap unit baru wajib menjalani Pre-Delivery Inspection sebelum diserahkan kepada kon...
3,KEB-AI-03,KEB-AI-03 Kebijakan Penggunaan Gen AI,"Karyawan dilarang memasukkan data pribadi konsumen, data keuangan rahasia, atau kode s..."


**Penjelasan kode**
- `DOKUMEN` — kumpulan dokumen internal (di produksi: dibaca dari PDF/Word/SharePoint). Kunci dict = kode & judul dokumen, dipakai sebagai **sumber sitasi**.
- `pecah_dokumen()` — *chunking* berbasis jumlah kata:
  - `maks_kata=60` — ukuran potongan. Terlalu besar → konteks tidak fokus & boros token; terlalu kecil → makna terpotong.
  - `tumpang=15` — 15 kata terakhir potongan sebelumnya diulang di awal potongan berikutnya (*overlap*), agar kalimat di perbatasan tidak kehilangan konteks.
  - `mulai += maks_kata - tumpang` — geser jendela sebesar 45 kata.
- `judul.split()[0]` — mengambil kode dokumen (mis. `SOP-GAR-01`) untuk sitasi ringkas.

In [11]:
STOPWORDS_ID = ["yang", "dan", "di", "ke", "dari", "untuk", "dengan", "atau", "pada", "wajib", "ini", "itu",
                "apa", "berapa", "bagaimana", "apakah", "saya", "kita", "harus", "dalam", "oleh", "sebelum"]

vektorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words=STOPWORDS_ID, sublinear_tf=True)
matriks = vektorizer.fit_transform(chunks["isi"])
print("Dimensi matriks TF-IDF (potongan × fitur):", matriks.shape)


def cari(pertanyaan, k=3, ambang=0.15):
    skor = cosine_similarity(vektorizer.transform([pertanyaan]), matriks).ravel()
    urut = skor.argsort()[::-1][:k]
    hasil = chunks.iloc[urut].assign(skor=skor[urut].round(3))
    return hasil[hasil["skor"] >= ambang]


cari("Berapa lama batas pengajuan klaim garansi?")

Dimensi matriks TF-IDF (potongan × fitur): (6, 400)


,sumber,judul,isi,skor
0,SOP-GAR-01,SOP-GAR-01 Klaim Garansi,Klaim garansi diajukan dealer melalui sistem klaim paling lambat 14 hari kalender sete...,0.311


**Penjelasan kode**
- **TF-IDF** (*Term Frequency – Inverse Document Frequency*): kata diberi bobot tinggi jika sering muncul di satu potongan tetapi jarang di potongan lain → kata yang **khas** menjadi penentu.
  - `ngram_range=(1, 2)` — memakai kata tunggal dan pasangan kata ("klaim garansi"), sehingga frasa lebih dikenali.
  - `stop_words=STOPWORDS_ID` — membuang kata umum bahasa Indonesia yang tidak informatif.
  - `sublinear_tf=True` — memakai `1 + log(tf)` agar kata yang diulang-ulang tidak mendominasi.
- `fit_transform` — mempelajari kosakata & menghasilkan matriks jarang (*sparse*) ukuran potongan × fitur.
- `cosine_similarity(vektor_pertanyaan, matriks)` — kemiripan sudut antar vektor (0 = tidak mirip, 1 = identik).
- `argsort()[::-1][:k]` — urutkan indeks dari skor tertinggi dan ambil *k* teratas.
- `ambang=0.15` — potongan dengan skor terlalu rendah dibuang agar LLM tidak diberi konteks yang tidak relevan. Nilai ambang ditentukan dari percobaan: pertanyaan relevan umumnya mendapat skor > 0,25, sedangkan pertanyaan di luar topik < 0,12.

In [12]:
def tanya_dokumen(pertanyaan, k=3, tampil=True):
    hasil = cari(pertanyaan, k=k)
    konteks = "\n".join(f"[SUMBER: {r.sumber}] {r.isi}" for r in hasil.itertuples()) or "(kosong)"
    prompt = f"""TUGAS: JAWAB_DARI_DOKUMEN
Anda asisten kebijakan internal. Jawab pertanyaan HANYA berdasarkan KONTEKS.
Sertakan kode sumber dalam kurung siku, contoh [SOP-GAR-01].
Jika jawaban tidak ada di KONTEKS, katakan tidak ditemukan — jangan menebak.
KONTEKS:
{konteks}
AKHIR_KONTEKS
PERTANYAAN: {pertanyaan} AKHIR_PERTANYAAN"""
    jawaban = llm.generate(prompt)
    if tampil:
        print(f"Q: {pertanyaan}\nA: {jawaban}\n   (potongan dipakai: {hasil['sumber'].tolist()})\n")
    return jawaban, hasil


for q in ["Berapa lama batas pengajuan klaim garansi setelah perbaikan?",
          "Bolehkah saya memasukkan data pribadi konsumen ke AI publik?",
          "Berapa lama keluhan urgensi tinggi harus ditindaklanjuti?",
          "Berapa harga saham perusahaan hari ini?"]:
    tanya_dokumen(q)

Q: Berapa lama batas pengajuan klaim garansi setelah perbaikan?
A: Klaim garansi diajukan dealer melalui sistem klaim paling lambat 14 hari kalender setelah perbaikan selesai. [SOP-GAR-01]
   (potongan dipakai: ['SOP-GAR-01'])

Q: Bolehkah saya memasukkan data pribadi konsumen ke AI publik?
A: Karyawan dilarang memasukkan data pribadi konsumen, data keuangan rahasia, atau kode sumber internal ke layanan AI publik. [KEB-AI-03]
   (potongan dipakai: ['KEB-AI-03'])

Q: Berapa lama keluhan urgensi tinggi harus ditindaklanjuti?
A: Keluhan dengan urgensi tinggi, yaitu yang menyangkut keselamatan atau motor tidak dapat digunakan, wajib ditindaklanjuti dalam 1x24 jam. [SOP-CR-05]
   (potongan dipakai: ['SOP-CR-05'])

Q: Berapa harga saham perusahaan hari ini?
A: Informasi tersebut tidak ditemukan dalam dokumen internal yang tersedia.
   (potongan dipakai: [])



**Penjelasan kode**
- `hasil.itertuples()` — iterasi baris DataFrame sebagai *named tuple* (`r.sumber`, `r.isi`), lebih cepat dari `iterrows`.
- Setiap potongan diberi label `[SUMBER: kode]` di dalam konteks → LLM dapat mengutip sumbernya.
- `or "(kosong)"` — jika tidak ada potongan yang lolos ambang, konteks bernilai "(kosong)".
- Instruksi kunci pada prompt RAG:
  1. *Jawab HANYA berdasarkan KONTEKS* → mencegah halusinasi.
  2. *Sertakan sumber* → jawaban bisa diverifikasi pembaca.
  3. *Jika tidak ada, katakan tidak ditemukan* → pertanyaan di luar dokumen (harga saham) tidak dijawab dengan tebakan.

### Evaluasi retrieval: apakah dokumen yang benar ditemukan?
Kualitas RAG sangat bergantung pada tahap pencarian. Kita ukur **Hit@1** (dokumen benar di peringkat 1) dan **Hit@3** (ada di 3 teratas) dengan sekumpulan pertanyaan uji.

In [13]:
uji_rag = [
    ("Dokumen apa saja yang wajib dilampirkan saat klaim?", "SOP-GAR-01"),
    ("Klaim di atas dua juta perlu persetujuan siapa?", "SOP-GAR-01"),
    ("Apa saja yang diperiksa saat PDI unit baru?", "SOP-PDI-02"),
    ("Siapa yang menandatangani checklist PDI?", "SOP-PDI-02"),
    ("Apakah keluaran AI harus ditinjau manusia?", "KEB-AI-03"),
    ("Kapan oli mesin sebaiknya diganti?", "PAN-SRV-04"),
    ("Kapan v-belt dan roller CVT diperiksa?", "PAN-SRV-04"),
    ("Ke mana keluhan dieskalasi jika dealer tidak bisa menyelesaikan?", "SOP-CR-05"),
]
baris = []
for q, benar in uji_rag:
    top = cari(q, k=3, ambang=0)["sumber"].tolist()
    baris.append({"pertanyaan": q, "dokumen_benar": benar, "top3": top,
                  "hit@1": top[:1] == [benar], "hit@3": benar in top})
eval_rag = pd.DataFrame(baris)
display(eval_rag)
print(f"Hit@1 = {eval_rag['hit@1'].mean():.0%} | Hit@3 = {eval_rag['hit@3'].mean():.0%}")

,pertanyaan,dokumen_benar,top3,hit@1,hit@3
0,Dokumen apa saja yang wajib dilampirkan saat klaim?,SOP-GAR-01,"[SOP-GAR-01, SOP-GAR-01, PAN-SRV-04]",True,True
1,Klaim di atas dua juta perlu persetujuan siapa?,SOP-GAR-01,"[SOP-GAR-01, SOP-GAR-01, PAN-SRV-04]",True,True
2,Apa saja yang diperiksa saat PDI unit baru?,SOP-PDI-02,"[SOP-PDI-02, SOP-GAR-01, KEB-AI-03]",True,True
3,Siapa yang menandatangani checklist PDI?,SOP-PDI-02,"[SOP-PDI-02, SOP-CR-05, PAN-SRV-04]",True,True
4,Apakah keluaran AI harus ditinjau manusia?,KEB-AI-03,"[KEB-AI-03, SOP-CR-05, PAN-SRV-04]",True,True
5,Kapan oli mesin sebaiknya diganti?,PAN-SRV-04,"[PAN-SRV-04, SOP-GAR-01, SOP-PDI-02]",True,True
6,Kapan v-belt dan roller CVT diperiksa?,PAN-SRV-04,"[PAN-SRV-04, SOP-PDI-02, SOP-CR-05]",True,True
7,Ke mana keluhan dieskalasi jika dealer tidak bisa menyelesaikan?,SOP-CR-05,"[SOP-CR-05, SOP-GAR-01, SOP-GAR-01]",True,True


Hit@1 = 100% | Hit@3 = 100%


**Penjelasan kode**
- `uji_rag` — pasangan (pertanyaan, dokumen yang seharusnya ditemukan) yang disusun oleh pemilik dokumen. Ini adalah **set uji** RAG.
- `top[:1] == [benar]` — benar jika peringkat pertama tepat; `benar in top` — benar jika ada di 3 besar.
- `.mean()` pada kolom boolean = persentase `True`.
- Jika Hit@3 rendah: perbaiki *chunking*, tambah sinonim/istilah, atau ganti ke model *embedding*. Jika Hit@3 tinggi tetapi jawaban salah: perbaiki **prompt**.

## Langkah 7 — Asisten data internal: memilih SQL atau dokumen
Satu pintu tanya-jawab: pertanyaan tentang **angka** diarahkan ke database (Text-to-SQL), pertanyaan tentang **aturan/prosedur** diarahkan ke dokumen (RAG). Semua interaksi dicatat untuk audit.

In [14]:
KATA_ANGKA = ["berapa jumlah", "total", "rasio", "rata-rata", "paling sering", "per model", "tertinggi"]
log_asisten = []


def asisten_internal(pertanyaan):
    jalur = "SQL" if any(k in pertanyaan.lower() for k in KATA_ANGKA) else "DOKUMEN"
    if jalur == "SQL":
        hasil = tanya_database(pertanyaan)
        ringkas = "ditolak" if hasil is None else f"{len(hasil)} baris"
    else:
        jawaban, _ = tanya_dokumen(pertanyaan, tampil=False)
        print(f"Q: {pertanyaan}\nA: {jawaban}\n")
        hasil, ringkas = jawaban, "jawaban teks"
    log_asisten.append({"waktu": time.strftime("%H:%M:%S"), "jalur": jalur,
                        "pertanyaan": samarkan_teks(pertanyaan), "hasil": ringkas})
    return hasil


display(asisten_internal("Komponen apa yang paling sering diklaim?"))
asisten_internal("Klaim garansi ditolak dalam kondisi apa saja?")
pd.DataFrame(log_asisten)

Q: Komponen apa yang paling sering diklaim?
SQL: SELECT komponen, COUNT(*) AS jumlah, ROUND(AVG(biaya)) AS rata_biaya FROM klaim_garansi WHERE komponen IS NOT NULL AND biaya > 0 GROUP BY komponen ORDER BY juml ...


,komponen,jumlah,rata_biaya
0,Pompa bensin,58,1259103.0
1,Injektor,57,1265474.0
2,Seal shock,52,1532481.0
3,Starter,47,1303234.0
4,Kampas kopling,44,1357000.0
5,ECU,43,1320349.0
6,Kiprok,39,1326077.0


Q: Klaim garansi ditolak dalam kondisi apa saja?
A: Klaim garansi ditolak apabila kerusakan disebabkan modifikasi, kecelakaan, atau penggunaan suku cadang tidak asli. [SOP-GAR-01]



,waktu,jalur,pertanyaan,hasil
0,15:19:57,SQL,Komponen apa yang paling sering diklaim?,7 baris
1,15:19:57,DOKUMEN,Klaim garansi ditolak dalam kondisi apa saja?,jawaban teks


**Penjelasan kode**
- Router berbasis kata kunci (`KATA_ANGKA`) — sederhana dan mudah dijelaskan. Alternatifnya: minta LLM memilih jalur seperti *router* di Materi 1.
- `log_asisten` menyimpan jalur & hasil; pertanyaan pun **disamarkan** (`samarkan_teks`) sebelum dicatat, karena pengguna bisa saja mengetik nomor HP konsumen di pertanyaannya.

## Ringkasan
| Tantangan data internal | Solusi di notebook ini |
|---|---|
| Kualitas data buruk | Profiling otomatis + catatan perbaikan oleh LLM (tanpa kirim data mentah) |
| Data pribadi | Tabel kebijakan, pseudonim SHA-256 + salt, masking regex, hapus kolom tak perlu |
| Data terstruktur | Text-to-SQL: skema terkurasi → validator → koneksi read-only |
| Dokumen / SOP | RAG: chunking + TF-IDF + prompt ber-sitasi + evaluasi Hit@k |
| Akuntabilitas | Log audit setiap pertanyaan & keputusan |

## Latihan
1. Tambahkan dokumen baru (mis. SOP pengembalian unit) ke `DOKUMEN`, bangun ulang indeks TF-IDF, dan buat 2 pertanyaan uji baru.
2. Ubah `maks_kata` menjadi 25 dan 100. Bagaimana pengaruhnya pada Hit@1?
3. Coba lolos dari validator SQL dengan `SELECT * FROM konsumen`. Mengapa masih tertahan / tidak tertahan? Perbaiki validator bila perlu (petunjuk: `*` mengambil semua kolom termasuk PII — pertimbangkan *view* tanpa PII).
4. Tambahkan pola penyamaran untuk nomor polisi kendaraan (mis. `B 1234 XYZ`).